# Эмбеддинги и векторный поиск

Превращаем чанки из урока 3 в векторы фиксированной длины (эмбеддинги),
чтобы искать по документации по смыслу, а не по совпадению слов.

## Пересобираем чанки

Функция `chunk_text()` из `chunking.ipynb` теперь живёт в общем модуле `rag.py`,
чтобы не копировать её из ноутбука в ноутбук. Импортируем оттуда все строительные блоки.


In [ ]:
from rag import (
    EMBED_MODEL,
    RagIndex,
    build_faiss_index,
    embed_texts,
    load_chunks,
    load_embed_model,
    vector_search,
)


In [ ]:
chunks = load_chunks("docs")

print(f"Всего чанков: {len(chunks)}")


## Считаем эмбеддинги для чанков

Модель `paraphrase-multilingual-MiniLM-L12-v2` — многоязычная (включая русский),
выдаёт векторы длиной 384. При первом запуске скачается с Hugging Face (~470 МБ),
дальше берётся из локального кеша.

`normalize_embeddings=True` приводит каждый вектор к единичной длине —
тогда косинусная близость эквивалентна скалярному произведению,
и с FAISS будет удобнее работать.

In [ ]:
import numpy as np

print(f"Загружаем модель эмбеддингов: {EMBED_MODEL}")
embed_model = load_embed_model()

# Берём только тексты чанков (метаданные оставим в chunks как есть)
texts = [c["text"] for c in chunks]

# Считаем эмбеддинги: на выходе - матрица (N, 384)
chunk_embeddings = embed_texts(texts, embed_model, show_progress_bar=True)

print(f"Размер матрицы эмбеддингов: {chunk_embeddings.shape}")


## Заглянем внутрь

Сами числа ничего не говорят — они имеют смысл только в сравнении друг с другом.
Норма равна единице, потому что мы запросили нормализацию векторов.

In [17]:
print(f"Первый чанк: {chunks[0]['text'][:10]}...")
print(f"Его эмбеддинг (первые 10 чисел): {chunk_embeddings[0][:10]}")
print(f"Длина вектора (норма): {np.linalg.norm(chunk_embeddings[0]):.4f}")

Первый чанк: # Document...
Его эмбеддинг (первые 10 чисел): [-0.0613074  -0.05510945 -0.03115823 -0.01005449  0.03433183  0.04836935
 -0.08265641  0.02022992 -0.00097118  0.04949648]
Длина вектора (норма): 1.0000


## Строим векторный индекс

Упаковываем эмбеддинги в FAISS-индекс. `IndexFlatIP` — «плоский» индекс
с метрикой Inner Product (скалярное произведение): он сравнивает запрос
со всеми векторами по очереди, без приближений. Для нескольких сотен чанков
этого достаточно.

Так как векторы нормализованы, скалярное произведение и есть косинусная близость.
`index.add()` принимает матрицу (N, dim) и добавляет каждую строку как отдельный вектор.

In [ ]:
index = build_faiss_index(chunk_embeddings)

print(f"В индексе {index.ntotal} векторов размерности {index.d}")


## Первый поиск

Функция `vector_search()` из `rag.py` повторяет алгоритм поиска по облаку точек:

1. Превращаем запрос в вектор **той же моделью**, что и чанки.
2. Просим у индекса top-k ближайших.
3. Для каждого попадания возвращаем текст чанка, его источник и оценку близости.

Модель для запроса всегда должна совпадать с моделью для чанков — векторы разных
моделей живут в разных пространствах, и расстояние между ними не имеет смысла.

Чанки, модель и индекс собираем в один объект `RagIndex`, который и передаём в поиск.
В других ноутбуках всё это делает одна функция `build_index("docs")`.


In [ ]:
rag = RagIndex(chunks=chunks, embed_model=embed_model, index=index)

vector_search(rag, "What is a Modelfile?", top_k=1)


## Проверяем на трёх вопросах

Для каждого вопроса выводим топ-3 чанков с источником и `score`.
Ожидание: топ-1 попадает в тематически правильный раздел документации,
а его `score` заметно выше, чем у второго и третьего.

Второй вопрос задан на русском, а документация на английском —
многоязычная модель эмбеддингов должна справиться.

In [ ]:
questions = [
    "What HTTP method does /api/generate use?",
    "Как сменить папку, где хранятся модели?",
    "What is a Modelfile?",
]

for q in questions:
    print(f"=== {q} ===")
    for i, r in enumerate(vector_search(rag, q, top_k=3), 1):
        print(f"#{i}  [{r['source']}]  score={r['score']:.3f}")
        print(f"    {r['text'][:120].strip()}...\n")
